# `<reference-id>` — predicted vs. measured

**Built by**: <name>  
**Date**: <YYYY-MM-DD>  
**Bench setup**: <Keysight E4990 + B-coil + FLIR T540 + Yokogawa GS610>  
**MagnaDesign version evaluated**: rendered automatically below.

Mandatory cells (do not delete or rename):

1. **load** — pulls `spec.pfc`, the engine catalogues, the
   `measurements.csv`, and `thresholds.yaml`.
2. **engine** — runs `design()` on the loaded spec.
3. **compare** — generates `MetricComparison` rows.
4. **plots** — overlay predicted-vs-measured curves
   (impedance sweep, B(t), thermal map). Notebook-author
   provides the plot code.
5. **summary** — PASS/FAIL verdict CI keys on. The last
   notebook cell **must** call `summary.all_passed` and
   raise an `AssertionError` if False, so papermill
   surfaces a regression as a non-zero exit code.

In [ ]:
import sys
from pathlib import Path

# Resolve the project root so `import validation.lib` works
# regardless of where papermill is invoked from.
_HERE = Path.cwd()
_ROOT = next(p for p in (_HERE, *_HERE.parents) if (p / "validation").is_dir())
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from pfc_inductor.data_loader import (
    ensure_user_data,
    load_cores,
    load_materials,
    load_wires,
)
from pfc_inductor.project import load_project
from validation.lib import (
    compare,
    load_measurements,
    load_thresholds,
    render_summary,
)

DESIGN_DIR = _HERE  # the notebook lives in validation/<id>/
REFERENCE_ID = DESIGN_DIR.name

ensure_user_data()
materials = load_materials()
cores = load_cores()
wires = load_wires()

project = load_project(DESIGN_DIR / "spec.pfc")
thresholds = load_thresholds(_ROOT / "validation" / "thresholds.yaml")
measurements = load_measurements(DESIGN_DIR / "measurements.csv")

print(f"reference: {REFERENCE_ID}")
print(f"topology : {project.spec.topology}")
print(f"measurements: {len(measurements.all)} rows")

In [ ]:
from pfc_inductor.design import design

material = next(m for m in materials if m.id == project.selection.material_id)
core = next(c for c in cores if c.id == project.selection.core_id)
wire = next(w for w in wires if w.id == project.selection.wire_id)

result = design(project.spec, core, wire, material)
print(f"L_actual = {result.L_actual_uH:.1f} uH")
print(f"B_pk     = {result.B_pk_T * 1000:.1f} mT")
print(f"T_winding= {result.T_winding_C:.1f} C")
print(f"P_total  = {result.losses.P_total_W:.2f} W")

In [ ]:
comparisons, summary = compare(result, measurements, thresholds)
report = render_summary(comparisons, summary, project_label=REFERENCE_ID)
print(report)

In [ ]:
# TODO: notebook-author overlay plots:
#   - L(f) impedance sweep — predicted curve from rolloff +
#     measurement points.
#   - B(t) at the operating point — engine waveform vs. B-coil capture.
#   - Thermal map — predicted T_winding line vs. FLIR profile.
#
# Each plot saves to ``DESIGN_DIR / 'plots'`` so GitHub Pages
# can embed without re-running the notebook.
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.set_title(f"{REFERENCE_ID} — example placeholder plot")
ax.text(
    0.5,
    0.5,
    "Replace with predicted-vs-measured plots",
    ha="center",
    va="center",
    transform=ax.transAxes,
)
ax.set_xticks([])
ax.set_yticks([])
plt.show()

In [ ]:
# CI hook — papermill exit-code is non-zero when this assert
# trips, so a release that regresses on a previously-PASS
# reference design fails the build.
assert summary.all_passed, (
    f"{REFERENCE_ID}: {len(summary.failures)} of {summary.n_total} metrics regressed.\n\n{report}"
)